In [57]:
# scrape eloratings.net

import pandas as pd
import numpy as np
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from io import StringIO

options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

driver.get("https://www.eloratings.net/")

try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, "grid-canvas"))
    )

    rows = driver.find_elements(By.CSS_SELECTOR, ".grid-canvas .slick-row")

    data = []
    for row in rows:
        cells = row.find_elements(By.CLASS_NAME, "slick-cell")
        row_data = [cell.text for cell in cells]
        data.append(row_data)

    columns = ['Rank', 'Team', 'Rating', 'Avg_Rank', 'Avg_Rating',
               '1yr_Rank_Change', '1yr_Rating_Change', 'Matches_Total',
               'Matches_Home', 'Matches_Away', 'Matches_Neutral',
               'Matches_Wins', 'Matches_Losses', 'Matches_Draws',
               'Goals_For', 'Goals_Against']

    df = pd.DataFrame(data
                      , columns=columns
                      )

    # Megjelenítés
    print(f"Talált sorok száma: {len(df)}")
    print("\nElső 10 sor:")
    print(df.head(10))
    
finally:
    driver.quit()

Talált sorok száma: 244

Első 10 sor:
  Rank         Team Rating Avg_Rank Avg_Rating 1yr_Rank_Change  \
0    1        Spain   2171        7       1945               0   
1    2    Argentina   2113        5       1985               0   
2    3       France   2062       16       1792               0   
3    4      England   2042        4       1982               0   
4    5     Colombia   1998       49       1617              +2   
5    6       Brazil   1978        4       1998              −1   
6    7     Portugal   1976       19       1796              −1   
7    8  Netherlands   1959       15       1847              +2   
8    9      Croatia   1933       12       1881              +5   
9    9      Ecuador   1933       64       1522              +3   

  1yr_Rating_Change Matches_Total Matches_Home Matches_Away Matches_Neutral  \
0                −6           778          338          302             138   
1                +5          1110          382          419             309  

In [83]:
# Simulate

import numpy as np
import pandas as pd
import itertools
from collections import defaultdict
import random


def simulate_match(teamA, teamB, df, knockout=False):
    """Meccs szimuláció: összes gól szétosztva ELO arány szerint."""

    eloA = float(df.loc[df["Team"] == teamA, "Rating"].values[0])
    eloB = float(df.loc[df["Team"] == teamB, "Rating"].values[0])

    # ELO arány
    pA = eloA / (eloA + eloB)
    pB = 1 - pA

    # összes gól generálása Poisson alapján
    avg_goals = 2.5
    total_goals = np.random.poisson(avg_goals)

    # szétosztás
    goals_A = int(round(total_goals * pA))
    goals_B = total_goals - goals_A

    # knockout: nincs döntetlen
    if knockout and goals_A == goals_B:
        # hosszabbítás / büntető ELO-súlyozással
        prob_A_win = pA
        winner = np.random.choice([teamA, teamB], p=[prob_A_win, 1-prob_A_win])
        if winner == teamA:
            goals_A += 1
        else:
            goals_B += 1

    return goals_A, goals_B

def generate_group_matches(teams):
    return list(itertools.combinations(teams, 2))

def rank_group_standings(df_standings, h2h_results):
    """FIFA tie-breaking rules implementálása."""

    # 1) Points → 2) overall GD → 3) overall GF
    df_sorted = df_standings.sort_values(
        by=["Points", "GD", "GF", "FairPlay", "FifaRank"],
        ascending=[False, False, False, False, True]
    ).reset_index(drop=True)

    # Head-to-head corrections (bonyolult logika nagy döntetlencsoportoknál)
    # Egyszerűsített, de FIFA-val kompatibilis újra-ellenőrzés
    changed = True
    while changed:
        changed = False
        for i in range(len(df_sorted)-1):
            t1 = df_sorted.loc[i, "Team"]
            t2 = df_sorted.loc[i+1, "Team"]

            # head-to-head létezik-e?
            if (t1, t2) in h2h_results:
                g1, g2 = h2h_results[(t1, t2)]
            elif (t2, t1) in h2h_results:
                g2, g1 = h2h_results[(t2, t1)]
            else:
                continue

            if g2 > g1:   # fordítva kellene lennie
                df_sorted.loc[i], df_sorted.loc[i+1] = df_sorted.loc[i+1].copy(), df_sorted.loc[i].copy()
                changed = True

    return df_sorted.reset_index(drop=True)

def simulate_group(group_name, teams, df):
    matches = generate_group_matches(teams)

    standings = pd.DataFrame({
        "Team": teams,
        "Points": 0,
        "GF": 0,
        "GA": 0,
        "GD": 0,
        "FairPlay": 0,
        "FifaRank": [df.loc[df["Team"] == t, "Rank"].values[0] for t in teams]
    })

    h2h = {}   # head-to-head eredmények (teamA, teamB): (gA, gB)

    for A, B in matches:
        gA, gB = simulate_match(A, B, df)

        h2h[(A, B)] = (gA, gB)

        # Update stats
        for team, GF, GA in [(A, gA, gB), (B, gB, gA)]:
            standings.loc[standings["Team"] == team, "GF"] += GF
            standings.loc[standings["Team"] == team, "GA"] += GA
            standings.loc[standings["Team"] == team, "GD"] = standings["GF"] - standings["GA"]

        # Points
        if gA > gB:
            standings.loc[standings["Team"] == A, "Points"] += 3
        elif gB > gA:
            standings.loc[standings["Team"] == B, "Points"] += 3
        else:
            standings.loc[standings["Team"] == A, "Points"] += 1
            standings.loc[standings["Team"] == B, "Points"] += 1

    ranked = rank_group_standings(standings, h2h)
    top2 = ranked.iloc[:2]["Team"].tolist()

    return ranked, top2

def simulate_knockout_round(team_pairs, df):
    winners = []
    for A, B in team_pairs:
        gA, gB = simulate_match(A, B, df, knockout=True)
        winners.append(A if gA > gB else B)
    return winners

def simulate_world_cup(groups, df):
    group_winners = {}
    group_runners = {}
    group_thirds = []

    # GROUP STAGE
    for gname, teams in groups.items():
        ranked, top2 = simulate_group(gname, teams, df)
        group_winners[gname] = top2[0]
        group_runners[gname] = top2[1]

        # harmadik helyezett mentése
        third_row = ranked.iloc[2].copy()
        third_row["Group"] = gname
        group_thirds.append(third_row)

    # ---- BEST THIRD-PLACED TEAMS (SELECT 8 OF 12) ----
    third_df = pd.DataFrame(group_thirds)

    # rendezés pontok → gólkülönbség → gólok → ha döntetlen, random
    def tie_breaker(sub_df):
        if len(sub_df) <= 1:
            return sub_df
        # ha teljesen azonos az első három érték, keverjük shuffle-lal
        vals = sub_df[["Points", "GD", "GF"]].values
        if np.all(vals[0] == vals):
            return sub_df.sample(frac=1).reset_index(drop=True)
        return sub_df

    # először a szokásos rendezés
    third_df = third_df.sort_values(
        by=["Points", "GD", "GF"], ascending=[False, False, False]
    ).reset_index(drop=True)

    # iteratívan ellenőrizve az azonos pont/gd/gf csapatokat
    grouped = third_df.groupby(["Points", "GD", "GF"], group_keys=False)
    third_df = grouped.apply(tie_breaker).reset_index(drop=True)

    best8_thirds = third_df.iloc[:8]["Team"].tolist()

    # ---- BUILD ROUND OF 32 TEAMS ----
    qualified_teams = []

    group_list = list(groups.keys())
    for i in range(0, len(group_list), 2):
        G1 = group_list[i]
        G2 = group_list[i+1]
        qualified_teams.append(group_winners[G1])
        qualified_teams.append(group_runners[G2])
        qualified_teams.append(group_winners[G2])
        qualified_teams.append(group_runners[G1])

    qualified_teams.extend(best8_thirds)

    # biztos 32 csapat
    assert len(qualified_teams) == 32, f"ERROR: qualified={len(qualified_teams)} not 32"

    # ---- CREATE R32 PAIRINGS ----
    round32_pairs = list(zip(qualified_teams[0::2], qualified_teams[1::2]))

    # ---- KNOCKOUT PHASE ----
    r32 = simulate_knockout_round(round32_pairs, df)

    r16_pairs = list(zip(r32[0::2], r32[1::2]))
    r16 = simulate_knockout_round(r16_pairs, df)

    qf_pairs = list(zip(r16[0::2], r16[1::2]))
    qf = simulate_knockout_round(qf_pairs, df)

    sf_pairs = list(zip(qf[0::2], qf[1::2]))
    semifinals = simulate_knockout_round(sf_pairs, df)

    final_pair = (semifinals[0], semifinals[1])
    champion = simulate_knockout_round([final_pair], df)[0]
    runner_up = final_pair[1] if champion == final_pair[0] else final_pair[0]
    # bronzmeccs
    losers = [t for sf in sf_pairs for t in sf if t != champion and t != runner_up]
    gB_A, gB_B = simulate_match(losers[0], losers[1], df, knockout=True)
    third = losers[0] if gB_A > gB_B else losers[1]
    fourth = losers[1] if third == losers[0] else losers[0]

    return champion, runner_up, third, fourth

# run simulation N times
runs = 100

groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czechia"],
    "B": ["Canada", "Switzerland", "Qatar", "Italy"],
    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
    "D": ["United States", "Australia", "Paraguay", "Turkey"],
    "E": ["Germany", "Ecuador", "Ivory Coast", "Curaçao"],
    "F": ["Netherlands", "Japan", "Tunisia", "Poland"],
    "G": ["Belgium", "Iran", "Egypt", "New Zealand"],
    "H": ["Spain", "Uruguay", "Saudi Arabia", "Cape Verde"],
    "I": ["France", "Senegal", "Norway", "Iraq"],
    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
    "K": ["Portugal", "Colombia", "Uzbekistan", "Jamaica"],
    "L": ["England", "Croatia", "Panama", "Ghana"]
}


n = 0
sims = []
while n < runs:
    winner, runner_up, third, fourth = simulate_world_cup(groups, df)
    sims.append({'winner': winner,
                 'runner_up': runner_up,
                 'third': third,
                 'fourth': fourth})

    n += 1
    if n % 5 == 0:
        print(f"{n}/100")

sims_df = pd.DataFrame(sims)

print(sims_df.groupby('winner').winner.count().sort_values(ascending=False))
print(sims_df.groupby('runner_up').winner.count().sort_values(ascending=False))
print(sims_df.groupby('third').winner.count().sort_values(ascending=False))

5/100
10/100
15/100
20/100
25/100
30/100
35/100
40/100
45/100
50/100
55/100
60/100
65/100
70/100
75/100
80/100
85/100
90/100
95/100
100/100
winner
Argentina        26
Spain            23
Brazil            7
England           7
France            6
Colombia          6
Netherlands       5
Switzerland       3
Ecuador           3
Senegal           2
Iran              2
Panama            2
Portugal          2
Canada            1
Curaçao           1
Japan             1
Paraguay          1
Turkey            1
United States     1
Name: winner, dtype: int64
runner_up
Brazil           13
Ecuador          10
Spain             9
England           8
Netherlands       7
Argentina         7
France            5
Turkey            5
Uruguay           5
Italy             4
Germany           4
Austria           4
Colombia          3
Switzerland       3
Portugal          2
Paraguay          2
Norway            2
Belgium           1
Croatia           1
Ivory Coast       1
Morocco           1
Mexico          